# Stage 4, tier 1: the mRFP synonymous control

Research plan item 5, the low-cost control run in parallel (Gantt 18; Overleaf Section 12.2, item 5). `mRFP_Expression.csv` holds synonymous recodings of one gene, the red fluorescent protein mRFP, expressed in *E. coli*: different codons, one protein. A protein encoder therefore sees the same input for every row and cannot predict anything, which makes it the negative control. A nucleotide encoder sees the codons, so whatever it predicts has to come from codon choice.

The question is narrower than "does the nucleotide encoder predict expression". Codon choice also changes simple, well-known quantities: GC content, GC at the third codon position, codon usage, and how strongly the mRNA folds, above all near the start codon. The protocol asks whether the nucleotide encoders predict expression **above** GC3, codon-use features and the structure proxies of Track A. If they do, the encoders carry synonymous-codon information beyond those summaries; if not, what they carry is those summaries. This tests representation specificity, not fusion.

Everything here reuses Stage 1's pooled embeddings of the same 1,000-row sample (seed 42), read from `Code/stage1/embeddings/` with their identity checked. No model is loaded and nothing needs a GPU. Probes are Stage 1's: a standardized ridge regression, five folds with identical sequences kept in one fold, on a pinned fold assignment and ten further ones.

---

## Setup

In [1]:
# Standard library
import json
import os
from pathlib import Path

# Data handling
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Shared project code in Code/shared_code; this notebook lives in Code/stage4
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import analysis, datasets, embeddings, sequences
from shared_code.encoders import ENCODERS
from shared_code.splits import grouped_folds

# ViennaRNA supplies the folding-energy controls: pip install ViennaRNA
!pip install ViennaRNA
import RNA

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("ViennaRNA", RNA.__version__)


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
ViennaRNA 2.7.2


### Settings

The sample and the probe settings are Stage 1's, so the embeddings it cached are reused rather than recomputed. Two folding windows: Track A's proxy, the Nussinov pairing fraction of the first 200 nucleotides, and the ViennaRNA minimum free energy of the first 60 nucleotides and of the whole sequence. The data hold the coding sequence only, so the start-codon window begins at the ATG; in *E. coli*, folding around the start codon is the best-known way synonymous codons change expression (Kudla et al., 2009).

The slow steps keep their results in `results/tier1_mrfp/` as they go, so an interrupted run continues where it stopped; delete those files to recompute from nothing.

In [2]:
CONTROL = "mrfp_expression"
N_ROWS, SEED = 1000, 42                       # Stage 1's control sample
EMBEDDING_DIR = Path("../stage1/embeddings/mrfp_expression_n1000_seed42")
PROBE_SEED = 42                               # Stage 1's pinned fold assignment
N_ASSIGNMENTS = 10                            # further assignments, seeds 0 to 9
TRACK_A_WINDOW = 200                          # Track A's Nussinov prefix
START_WINDOW = 60                             # nucleotides from the start codon

RESULTS_DIR = Path("results") / "tier1_mrfp"
STRUCTURE_CACHE = RESULTS_DIR / "structure_features.csv"
PROBE_CACHE = RESULTS_DIR / "probe_scores.csv"
GAIN_CACHE = RESULTS_DIR / "gains_over_controls.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

control = datasets.load_dataset(CONTROL, n_rows=N_ROWS, seed=SEED)
seqs = control.sequences
y = control.labels.astype(float)
groups = control.groups                       # identical sequences share a fold
MODALITY = {e.key: e.modality for e in ENCODERS.values()}
LABEL = {e.key: e.label for e in ENCODERS.values()}
EMBEDDINGS = {key: embeddings.read_saved(encoder, seqs, EMBEDDING_DIR)
              for key, encoder in ENCODERS.items()}
print(f"{len(seqs)} rows ({control.n_sampled} sampled), {len(set(seqs))} distinct sequences, "
      f"lengths {sorted({len(s) for s in seqs})}, expression {y.min():.2f} to {y.max():.2f}")
print(f"{len(EMBEDDINGS)} cached embeddings read: "
      + ", ".join(f"{k} {v.shape[1]}" for k, v in EMBEDDINGS.items()))


def save(frame, path):
    """Write a table so that an interruption leaves the previous complete copy."""
    partial = path.with_name(path.name + ".partial")
    frame.to_csv(partial, index=False)
    os.replace(partial, path)

1000 rows (1000 sampled), 997 distinct sequences, lengths [678], expression 7.62 to 11.38
12 cached embeddings read: nt500m 1280, ntv2_100m 512, dnabert2 768, hyenadna 256, rnafm 640, rinalmo 640, mrnafm 1280, calm 768, esm2_8m 320, esm2_35m 480, esm2_150m 640, protbert 1024


---

## The pipeline control

Stage 1 already checked this (Overleaf Section 7): one protein, protein embeddings that barely move, nucleotide embeddings that do. Recomputed here from the same cache, because the rest of the notebook rests on it.

In [3]:
proteins = {sequences.translate_cds(s) for s in seqs}
spread = pd.DataFrame({
    "modality": [MODALITY[k] for k in EMBEDDINGS],
    "width": [EMBEDDINGS[k].shape[1] for k in EMBEDDINGS],
    "mean distance to centroid": [analysis.centroid_spread(EMBEDDINGS[k]) for k in EMBEDDINGS],
}, index=[LABEL[k] for k in EMBEDDINGS])
print(f"distinct proteins: {len(proteins)}")
spread.round(5)

distinct proteins: 1


,modality,width,mean distance to centroid
NT 500M human-ref,DNA,1280,4.72361
NT v2 100M multi-species,DNA,512,1.26535
DNABERT-2,DNA,768,0.74851
HyenaDNA large,DNA,256,0.61138
RNA-FM,RNA,640,0.81823
RiNALMo 150M,RNA,640,3.96281
mRNA-FM,RNA,1280,1.74951
CaLM,RNA,768,1.99879
ESM-2 8M,protein,320,0.00004
ESM-2 35M,protein,480,0.00004


---

## The controls

Every control is a property of the nucleotide sequence alone. The amino-acid composition and the length are the same for every row, so they are left out.

In [4]:
def structure_row(seq):
    rna = sequences.dna_to_rna(seq)
    return {"sequence": seq,
            "nussinov_200": sequences.nussinov_pairing_fraction(seq[:TRACK_A_WINDOW]),
            "mfe_start_60": RNA.fold(rna[:START_WINDOW])[1],
            "mfe_full": RNA.fold(rna)[1]}


# Resumable: Nussinov is cubic in the window and ViennaRNA folds 678 nucleotides per row.
done = pd.read_csv(STRUCTURE_CACHE) if STRUCTURE_CACHE.exists() else pd.DataFrame()
have = set(done["sequence"]) if len(done) else set()
todo = [s for s in dict.fromkeys(seqs) if s not in have]
for start in range(0, len(todo), 50):
    chunk = pd.DataFrame([structure_row(s) for s in todo[start:start + 50]])
    done = pd.concat([done, chunk], ignore_index=True)
    save(done, STRUCTURE_CACHE)
structure = done.set_index("sequence").loc[seqs].reset_index(drop=True)
print(f"structure features: {len(todo)} computed now, {len(seqs) - len(todo)} read back")

codon_use = np.array([sequences.composition_features(s)[3:3 + 64] for s in seqs])
FEATURES = pd.DataFrame({
    "GC": [sequences.gc_content(s) for s in seqs],
    "GC3": [sequences.gc3_content(s) for s in seqs],
    "Nussinov, first 200 nt": structure["nussinov_200"],
    "MFE, first 60 nt": structure["mfe_start_60"],
    "MFE, whole sequence": structure["mfe_full"],
})
CONTROLS = {
    "GC": FEATURES[["GC"]].to_numpy(),
    "GC3": FEATURES[["GC3"]].to_numpy(),
    "codon use (64)": codon_use,
    "Nussinov, first 200 nt": FEATURES[["Nussinov, first 200 nt"]].to_numpy(),
    "MFE, first 60 nt": FEATURES[["MFE, first 60 nt"]].to_numpy(),
    "MFE, whole sequence": FEATURES[["MFE, whole sequence"]].to_numpy(),
}
# The protocol's set (GC3, codon use, Track A's structure proxy), and that set with GC and the
# two folding energies added.
CONTROL_SETS = {
    "Track A controls": np.hstack([CONTROLS["GC3"], codon_use, CONTROLS["Nussinov, first 200 nt"]]),
    "all controls": np.hstack([CONTROLS["GC"], CONTROLS["GC3"], codon_use,
                               CONTROLS["Nussinov, first 200 nt"], CONTROLS["MFE, first 60 nt"],
                               CONTROLS["MFE, whole sequence"]]),
}
pd.DataFrame({
    "Spearman with expression": [spearmanr(FEATURES[c], y).statistic for c in FEATURES],
    "min": FEATURES.min(), "max": FEATURES.max(),
}).round(3)

structure features: 997 computed now, 3 read back


,Spearman with expression,min,max
GC,-0.173,0.445,0.546
GC3,-0.249,0.376,0.677
"Nussinov, first 200 nt",0.132,0.650,0.810
"MFE, first 60 nt",0.264,-21.800,-2.800
"MFE, whole sequence",-0.144,-229.400,-155.900


---

## How well each representation predicts expression

Held-out R² of a ridge probe on each encoder's pooled embedding, and on each control. "Pinned" is Stage 1's fold assignment; the other two columns are the mean and spread of the five-fold mean over ten further assignments, which shows how much a value depends on the partition.

The protein encoders are expected near zero or below. Their embeddings differ between rows only by numerical noise, which the probe's standardization scales up to unit variance; a negative R² is what a probe fit to noise gives.

A random-feature null, 640 Gaussian columns, calibrates the scale: it is what a probe of that width gets from nothing.

In [5]:
rng = np.random.default_rng(0)
NULL = rng.normal(size=(len(seqs), 640))
MODELS = {**{LABEL[k]: X for k, X in EMBEDDINGS.items()},
          **CONTROLS, **CONTROL_SETS, "random features (640)": NULL}
KIND = {**{LABEL[k]: MODALITY[k] for k in EMBEDDINGS},
        **{c: "control" for c in (*CONTROLS, *CONTROL_SETS)}, "random features (640)": "null"}
SEEDS = (PROBE_SEED, *range(N_ASSIGNMENTS))

# Resumable, one model at a time.
scores = pd.read_csv(PROBE_CACHE) if PROBE_CACHE.exists() else pd.DataFrame()
for name, X in MODELS.items():
    if len(scores) and (scores["model"] == name).sum() == len(SEEDS):
        continue
    rows = [{"model": name, "seed": s,
             "r2": analysis.probe_scores(X, y, groups=groups, seed=s).mean()} for s in SEEDS]
    scores = pd.concat([scores[scores["model"] != name] if len(scores) else scores,
                        pd.DataFrame(rows)], ignore_index=True)
    save(scores, PROBE_CACHE)

pinned = scores[scores["seed"] == PROBE_SEED].set_index("model")["r2"]
further = scores[scores["seed"] != PROBE_SEED].groupby("model")["r2"]
PROBES = pd.DataFrame({"kind": pd.Series(KIND), "R2, pinned": pinned,
                       "R2, 10 assignments": further.mean(),
                       "SD over assignments": further.std(ddof=0)}).loc[list(MODELS)]
PROBES.round(3)

,kind,"R2, pinned","R2, 10 assignments",SD over assignments
NT 500M human-ref,DNA,0.296,0.291,0.013
NT v2 100M multi-species,DNA,0.281,0.264,0.016
DNABERT-2,DNA,0.226,0.206,0.012
HyenaDNA large,DNA,0.249,0.247,0.021
RNA-FM,RNA,0.222,0.223,0.008
RiNALMo 150M,RNA,0.285,0.284,0.019
mRNA-FM,RNA,0.463,0.466,0.013
CaLM,RNA,0.298,0.304,0.021
ESM-2 8M,protein,-0.003,-0.005,0.002
ESM-2 35M,protein,-0.003,-0.005,0.002


---

## What the encoders add above the controls

For each fold, the controls are fit first, then the encoder is fit to what the controls leave unexplained, and the two predictions are added on the held-out fold. The controls' residual on the training rows comes from an inner cross-validation, so the encoder learns from the controls' honest errors rather than their in-sample fit. The gain is the held-out R² of controls plus encoder minus that of the controls alone, on the same folds, over the ten further fold assignments.

This is stricter than concatenating the encoder with the controls in one ridge regression, where 66 control columns would share one penalty with up to 1,280 embedding columns and be shrunk with them.

In [6]:
def ridge():
    return Pipeline([("scale", StandardScaler()), ("model", RidgeCV(alphas=np.logspace(-3, 5, 20)))])


def gain_over(C, X, seed):
    """Held-out R^2 of the controls alone and of the controls plus X, on the same folds."""
    base, stacked = [], []
    for train, test in grouped_folds(groups, seed=seed):
        inner_groups = [groups[i] for i in train]
        inner = grouped_folds(inner_groups, seed=f"inner-{seed}")
        honest = cross_val_predict(ridge(), C[train], y[train], cv=inner)
        controls = ridge().fit(C[train], y[train]).predict(C[test])
        extra = ridge().fit(X[train], y[train] - honest).predict(X[test])
        base.append(r2_score(y[test], controls))
        stacked.append(r2_score(y[test], controls + extra))
    return float(np.mean(base)), float(np.mean(stacked))


CANDIDATES = {**{LABEL[k]: X for k, X in EMBEDDINGS.items()}, "random features (640)": NULL}

gains = pd.read_csv(GAIN_CACHE) if GAIN_CACHE.exists() else pd.DataFrame()
for set_name, C in CONTROL_SETS.items():
    for name, X in CANDIDATES.items():
        if len(gains) and ((gains["model"] == name) & (gains["controls"] == set_name)).sum() == N_ASSIGNMENTS:
            continue
        rows = []
        for s in range(N_ASSIGNMENTS):
            base, stacked = gain_over(C, X, s)
            rows.append({"model": name, "controls": set_name, "seed": s,
                         "controls R2": base, "with encoder R2": stacked})
        gains = pd.concat([gains, pd.DataFrame(rows)], ignore_index=True)
        save(gains, GAIN_CACHE)

gains["gain"] = gains["with encoder R2"] - gains["controls R2"]
GAINS = (gains.groupby(["controls", "model"], sort=False)
         .agg(**{"controls R2": ("controls R2", "mean"), "with encoder R2": ("with encoder R2", "mean"),
                 "gain": ("gain", "mean"), "lowest": ("gain", "min"), "highest": ("gain", "max"),
                 "assignments above 0": ("gain", lambda g: int((g > 0).sum()))}))
GAINS.insert(0, "kind", [KIND[m] for _, m in GAINS.index])
GAINS.round(3)

kind  controls R2  with encoder R2   gain  lowest  highest  assignments above 0
controls         model                                                                                                       
Track A controls NT 500M human-ref             DNA        0.190            0.320  0.130   0.118    0.146                   10
                 NT v2 100M multi-species      DNA        0.190            0.251  0.061   0.025    0.079                   10
                 DNABERT-2                     DNA        0.190            0.233  0.044   0.028    0.058                   10
                 HyenaDNA large                DNA        0.190            0.266  0.077   0.048    0.098                   10
                 RNA-FM                        RNA        0.190            0.244  0.054   0.041    0.065                   10
                 RiNALMo 150M                  RNA        0.190            0.287  0.098   0.068    0.120                   10
                 mRNA-FM                       RNA        0.190            0.451  0.261   0.231    0.290                   10
                 CaLM                          RNA        0.190            0.282  0.093   0.064    0.121                   10
                 ESM-2 8M                  protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 ESM-2 35M                 protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 ESM-2 150M                protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 ProtBERT                  protein        0.190            0.189 -0.000  -0.000    0.000                    3
                 random features (640)        null        0.190            0.188 -0.001  -0.003    0.000                    1
all controls     NT 500M human-ref             DNA        0.306            0.381  0.076   0.067    0.088                   10
                 NT v2 100M multi-species      DNA        0.306            0.334  0.029   0.010    0.048                   10
                 DNABERT-2                     DNA        0.306            0.332  0.026   0.010    0.034                   10
                 HyenaDNA large                DNA        0.306            0.342  0.036   0.005    0.057                   10
                 RNA-FM                        RNA        0.306            0.327  0.021   0.011    0.030                   10
                 RiNALMo 150M                  RNA        0.306            0.355  0.049   0.033    0.067                   10
                 mRNA-FM                       RNA        0.306            0.488  0.182   0.168    0.203                   10
                 CaLM                          RNA        0.306            0.328  0.022  -0.001    0.043                    9
                 ESM-2 8M                  protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 ESM-2 35M                 protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 ESM-2 150M                protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 ProtBERT                  protein        0.306            0.306 -0.000  -0.000    0.000                    3
                 random features (640)        null        0.306            0.305 -0.001  -0.004    0.001                    2

## Reading

**The control holds.** One protein, protein embeddings that differ only by numerical noise, nucleotide embeddings that spread; the spreads match Stage 1's to the third decimal, so these are the embeddings Stage 1 wrote. The four protein encoders probe at $R^2 = -0.005$, the same as 640 random columns, and add exactly nothing above the controls. Everything below is nucleotide-level information by construction.

**Every nucleotide encoder predicts expression above the Track A controls**, in every one of the ten fold assignments. GC3, the 64 codon frequencies and the Nussinov pairing fraction together explain 0.19 of the variance; each encoder adds between +0.04 (DNABERT-2) and +0.26 (mRNA-FM) on top of them. Adding GC and the two folding energies raises the controls to 0.31 and shrinks the gains, but seven of the eight encoders still gain in ten of ten assignments and CaLM in nine. So the plan item's question has its answer: the nucleotide encoders carry synonymous-codon information that the composition and structure summaries do not, and the Stage 2 claim that codon choice is real information for the codon-level encoders is supported on the one dataset built to test it.

**mRNA-FM is the standout.** Alone it reaches 0.47, twice any other encoder, and it keeps +0.18 above the fullest set of controls where the next best, NT 500M, keeps +0.08. RNA-FM, the RNA encoder BioLangFusion fuses, adds the least of the eight (+0.02). Stage 1 found mRNA-FM the least composition-dominated of the twelve encoders, and this is the task where that matters: within one gene, composition is nearly all a pooled embedding has to vary with, unless the encoder reads codon context. This is a reason to watch the mRNA-FM and CaLM swaps in Phase B.

**The controls rank themselves.** Codon use carries most of what simple features carry (0.18). The folding energy of the first 60 nucleotides is next (0.11, Spearman +0.26: a weaker fold after the start codon goes with higher expression), which is Kudla et al.'s finding on this dataset. GC3 alone carries 0.03, GC 0.01, and Track A's own structure proxy, the Nussinov pairing fraction, nothing (0.00); the whole-sequence folding energy nothing either. A structure control on this task has to be the start-codon window, not a global proxy.

**Limits.** One gene, so this measures within-gene codon choice only; 1,000 of the 1,459 rows, Stage 1's sample; pooled embeddings and linear probes, so a trained head on token-level streams could rank the encoders differently; and the controls are linear in the codon frequencies, so a nonlinear control (gradient-boosted trees on the same features, as the baseline hierarchy allows) is the next check before the gain is called information beyond codon usage. The Cambray follow-up data are worth raising now that this pilot gave a result.
